<a href="https://colab.research.google.com/github/Lornatt88/maai901-vision-project/blob/main/notebooks/03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import kagglehub
from google.colab import drive

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/maai901_skin"
test_df = pd.read_csv(os.path.join(save_dir, "test.csv"))
classes = ["nv", "bkl", "mel", "bcc"]
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Ready. Test images:", len(test_df), "| Running on:", device)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Mounted at /content/drive
Ready. Test images: 544 | Running on: cpu


In [6]:
!pip install -q gradio

import gradio as gr
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoModelForImageClassification

demo_model = AutoModelForImageClassification.from_pretrained(
    "microsoft/swin-base-patch4-window7-224", num_labels=4, ignore_mismatched_sizes=True)
demo_model.load_state_dict(torch.load(os.path.join(save_dir, "swin_finetuned.pt"), map_location=device))
demo_model.to(device).eval()

demo_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

labels = ["Mole (nv) - benign", "Benign keratosis (bkl) - benign",
          "Melanoma (mel) - malignant", "Basal cell carcinoma (bcc) - malignant"]
T = 0.25

def predict(img):
    x = demo_tf(img.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        p = F.softmax(demo_model(x).logits, dim=1)[0].cpu().numpy()
    cancer = p[2] + p[3]
    if cancer >= T:
        verdict = f"Possible skin cancer ({cancer:.0%} combined chance). This lesion should be checked by a dermatologist."
    else:
        verdict = f"Likely benign ({1 - cancer:.0%} chance). This is not a diagnosis. See a doctor if you are concerned."
    return {labels[i]: float(p[i]) for i in range(4)}, verdict

examples = []
for c in classes:
    examples += test_df[test_df["dx"] == c].sample(3, random_state=7)["path"].tolist()

theme = gr.themes.Soft(
    primary_hue="teal",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
)

force_light = """
function forceLight() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'light') {
        url.searchParams.set('__theme', 'light');
        window.location.href = url.href;
    }
}
"""

def build_interface(**style):
    return gr.Interface(
        fn=predict,
        inputs=gr.Image(type="pil", label="Upload a dermatoscopic image"),
        outputs=[gr.Label(num_top_classes=4, label="Probabilities"), gr.Textbox(label="Result")],
        title="Skin Lesion Check (research demo)",
        description="Swin-Base fine-tuned on HAM10000 (4 classes). Flags possible cancer at a 0.25 threshold. For research and education only, not a medical device.",
        examples=examples,
        examples_per_page=12,
        **style,
    )

try:
    demo = build_interface(theme=theme, js=force_light)
    demo.launch(share=True, inline=False)
except TypeError:
    demo = build_interface()
    demo.launch(share=True, inline=False, theme=theme, js=force_light)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-base-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([4, 1024])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([4])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, js. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bfa014f645b5eff846.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import shutil
ex_dir = os.path.join(save_dir, "examples")
os.makedirs(ex_dir, exist_ok=True)
for i, c in enumerate(classes):
    src = test_df[test_df["dx"] == c].iloc[0]["path"]
    shutil.copy(src, os.path.join(ex_dir, f"{i+1}_{c}.jpg"))
print(os.listdir(ex_dir))

['1_nv.jpg', '2_bkl.jpg', '3_mel.jpg', '4_bcc.jpg']
